# 01 — Data Exploration

Load raw data from the three primary authoritative sources and perform
basic visual/statistical exploration scoped to the **Mississauga pilot bbox**.

| Source | Class | Description |
|--------|-------|-------------|
| OEB Service Areas | `OEBFetcher` | LDC polygon boundaries |
| OSM Power Features | `OSMFetcher` | Substations, lines, poles, towers |
| IESO TX Registry | `IESOFetcher` | Transmission lines & substation ratings |

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box

from src.ingestion.oeb_fetcher import OEBFetcher
from src.ingestion.osm_fetcher import OSMFetcher
from src.ingestion.ieso_fetcher import IESOFetcher
from src.utils.config_loader import load_settings

cfg = load_settings()
bbox_cfg = cfg["region"]["bbox"]

# Mississauga pilot bounding box (EPSG:4326)
WEST  = bbox_cfg["west"]
SOUTH = bbox_cfg["south"]
EAST  = bbox_cfg["east"]
NORTH = bbox_cfg["north"]
BBOX  = (WEST, SOUTH, EAST, NORTH)

print(f"Pilot area bbox: W={WEST}  S={SOUTH}  E={EAST}  N={NORTH}")

## 1.1  OEB Distributor Service Areas

In [ ]:
oeb = OEBFetcher()
service_areas = oeb.fetch_service_areas()

print(f"Total LDC polygons (province-wide): {len(service_areas)}")
print(f"CRS: {service_areas.crs}")
print()
print(service_areas[["ldc_name", "ldc_id"]].head(10).to_string(index=False))

In [ ]:
# Clip to pilot bbox
bbox_poly = box(WEST, SOUTH, EAST, NORTH)
service_areas_clip = service_areas[service_areas.intersects(bbox_poly)].copy()

print(f"LDCs intersecting Mississauga bbox: {len(service_areas_clip)}")
print()
print("LDC list in pilot area:")
for _, row in service_areas_clip.iterrows():
    area_km2 = row.geometry.area * (111.32 ** 2)  # rough deg->km^2
    print(f"  {row['ldc_id']:6s}  {row['ldc_name']}  (~{area_km2:.0f} km\u00b2 clipped)")

## 1.2  OSM Power Features

In [ ]:
osm = OSMFetcher()
osm_power = osm.fetch_power_features(bbox=BBOX)

print(f"Total OSM power features in bbox: {len(osm_power)}")
print()
counts = osm_power["power"].value_counts()
print("Feature counts by power= tag:")
print(counts.to_string())

In [ ]:
# Voltage distribution for lines and cables
lines = osm_power[osm_power["power"].isin(["line", "minor_line", "cable"])].copy()
lines["voltage_kv"] = pd.to_numeric(lines.get("voltage", pd.Series(dtype=str)), errors="coerce") / 1000

print(f"OSM lines / cables: {len(lines)}")
print(f"  with voltage tag: {lines['voltage_kv'].notna().sum()}")
print()
print("Voltage tier breakdown (kV):")
print(
    lines["voltage_kv"]
    .dropna()
    .value_counts(bins=[0, 1, 15, 30, 120, 250, 600])
    .to_string()
)

## 1.3  IESO Transmission Facility Registry

In [ ]:
ieso = IESOFetcher()
tx_registry = ieso.fetch_transmission_registry()

print(f"IESO TX facilities (province-wide): {len(tx_registry)}")
print(f"Columns: {list(tx_registry.columns)}")
print()
print(tx_registry[["facility_name", "voltage_kv", "rating_mva", "operator"]]
      .sort_values("voltage_kv", ascending=False)
      .head(12)
      .to_string(index=False))

In [ ]:
# Clip IESO TX to pilot bbox
tx_clip = tx_registry[tx_registry.intersects(bbox_poly)].copy()

print(f"TX facilities in Mississauga bbox: {len(tx_clip)}")
print()
print("By voltage tier:")
print(tx_clip["voltage_kv"].value_counts().to_string())

## 1.4  Combined Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle("Ontario Grid Mapper — Raw Data Layers (Mississauga bbox)", fontsize=14, fontweight="bold")

# Panel 1: OEB service area polygons
ax = axes[0]
service_areas_clip.plot(
    ax=ax, column="ldc_name", legend=False,
    alpha=0.55, edgecolor="#333333", linewidth=0.8,
    cmap="tab20"
)
ax.set_title(f"OEB Service Areas\n({len(service_areas_clip)} LDCs in bbox)", fontsize=11)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(WEST, EAST)
ax.set_ylim(SOUTH, NORTH)

# Panel 2: OSM power features
ax = axes[1]
color_map = {
    "line":         "#e74c3c",
    "minor_line":   "#e67e22",
    "cable":        "#9b59b6",
    "substation":   "#2980b9",
    "transformer":  "#27ae60",
    "pole":         "#7f8c8d",
    "tower":        "#2c3e50",
}
legend_handles = []
for power_type, color in color_map.items():
    subset = osm_power[osm_power["power"] == power_type]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, markersize=3, linewidth=0.9, alpha=0.8)
        legend_handles.append(mpatches.Patch(color=color, label=f"{power_type} ({len(subset)})"))
ax.legend(handles=legend_handles, fontsize=7, loc="lower left")
ax.set_title(f"OSM Power Features\n({len(osm_power)} total)", fontsize=11)
ax.set_xlim(WEST, EAST)
ax.set_ylim(SOUTH, NORTH)

# Panel 3: IESO TX lines by voltage
ax = axes[2]
tx_lines = tx_clip[tx_clip.geometry.geom_type.isin(["LineString", "MultiLineString"])]
if len(tx_lines) > 0:
    tx_lines.plot(
        ax=ax, column="voltage_kv", cmap="YlOrRd",
        linewidth=1.5, legend=True,
        legend_kwds={"label": "Voltage (kV)", "shrink": 0.6}
    )
tx_pts = tx_clip[tx_clip.geometry.geom_type == "Point"]
if len(tx_pts) > 0:
    tx_pts.plot(ax=ax, color="navy", markersize=10, zorder=5, marker="s")
ax.set_title(f"IESO TX Registry\n({len(tx_clip)} facilities in bbox)", fontsize=11)
ax.set_xlim(WEST, EAST)
ax.set_ylim(SOUTH, NORTH)

plt.tight_layout()
plt.savefig("../data/outputs/01_data_exploration_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to data/outputs/01_data_exploration_map.png")

## 1.5  Summary Statistics

In [ ]:
summary = {
    "OEB LDCs (province)": len(service_areas),
    "OEB LDCs (pilot bbox)": len(service_areas_clip),
    "OSM power features (bbox)": len(osm_power),
    "  — lines / cables": int(osm_power["power"].isin(["line", "minor_line", "cable"]).sum()),
    "  — substations": int((osm_power["power"] == "substation").sum()),
    "  — poles / towers": int(osm_power["power"].isin(["pole", "tower"]).sum()),
    "IESO TX facilities (province)": len(tx_registry),
    "IESO TX facilities (bbox)": len(tx_clip),
}

print("=" * 48)
print("  DATA EXPLORATION SUMMARY")
print("=" * 48)
for k, v in summary.items():
    print(f"  {k:<38} {v:>6}")
print("=" * 48)